# Open Notebook in Colab


[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorial_v3.ipynb)

## ⚠️ Important: Data Privacy and Anonymization

Before using this tutorial with your own data, you should ensure that any sensitive or personally identifiable information (PII) has been removed or anonymized.

Because this package sends text to an LLM API for analysis, **any raw text you provide may leave your local environment** depending on your configuration. For this reason, you should **not upload identifiable or sensitive data unless it has been properly anonymized first**.

### A note on API data retention

Most companies will not use data you submit via an API (like in this tutorial) for training, and will delete the data after processing it within 30 days (e.g., OpenAI's policy: https://developers.openai.com/api/docs/guides/your-data). Policies vary by provider — check the terms of the model you select on OpenRouter — but in general, API submissions are treated more privately than chat-product submissions. This does **not** remove your responsibility to anonymize sensitive data first; it just means that, when working with properly anonymized text, API analysis is generally safe.

### Recommended approaches

If you are working with real data, we strongly recommend anonymizing it locally before running the analysis. We recommend:

- https://github.com/microsoft/presidio
- https://github.com/childmindresearch/anonymize-pii

These tools can help remove names, locations, emails, and other identifying information before any LLM processing occurs.

### If you prefer not to use real data

If you are not comfortable anonymizing your own dataset, you can use the **publicly available Reddit data provided in this repository**, which is safe to use for demonstration purposes.

---

**In short:**
✔ Use anonymized data whenever possible
✔ Be aware of your provider's API data retention policy
✔ When in doubt, use the provided sample dataset

In [ ]:
!pip install "git+https://github.com/childmindresearch/llm_tracker.git#egg=llm-tracker[tutorials]"

# LLM Tracker Tutorial — V3

**LLM Tracker** is a Python package for identifying instances of psychological constructs in qualitative text data — such as clinical documents, therapy transcripts, or open-ended survey responses — using large language models (LLMs).

The core workflow has four steps:

1. **LLM Coding** — run a language model over a set of documents to identify and extract quotes that are instances of psychological constructs defined in a codebook.
2. **Human Coding** — produce a second set of codings to compare against. In production this is real human-coded data; for testing purposes a second LLM run can serve as a stand-in.
3. **Comparison** — align the two sets of codings instance-by-instance using an LLM matcher, classifying each as a true positive, false positive, or false negative.
4. **Summary Tables** — compute pooled, per-document, and weighted performance metrics across constructs.

This V3 tutorial is split into two parts:

- **Part 1 — Dummy Data Workflow:** runs the LLM twice over the bundled sample CSV (`reddit_autism_anxiety_depression.csv`), using the second run as a stand-in for human coding. Useful for end-to-end pipeline testing without needing real human data.
- **Part 2 — Real Human Data Workflow:** the typical research workflow, where the LLM is run once and compared against externally produced human-coded data loaded from a CSV/xlsx file.

**Prerequisites:**
- An [OpenRouter](https://openrouter.ai) account and API key. OpenRouter provides access to a wide range of models (GPT-4o, Gemini, Claude, Llama, etc.) through a single API. Create an account, add credits, and generate a key at https://openrouter.ai/settings/keys.
- The sample data and codebook bundled with this repo (used in Part 1).
- For Part 2 only: your own human-coded export (CSV/xlsx).

## Imports

Import the core classes and functions used throughout this tutorial:
- `LLMTrackerAnalyzer` - runs LLM coding over a directory of document files or a CSV
- `LLMTrackerComparer` - aligns and compares two sets of codings (human vs LLM)
- `compute_summary_tables` - computes per-document, concatenated, and weighted summary metrics
- `format_concatenated`, `format_weighted_summary` - pretty-print the pooled and weighted summary tables
- `format_coding_table` - display analyzer results as a row-level table
- `load_human_coding` - load human-coded data from a CSV, TSV, or xlsx file


In [1]:
import pandas as pd
from llm_tracker import AnalyzerConfig, LLMTrackerAnalyzer
from llm_tracker.comparison import (
    LLMTrackerComparer,
    compute_summary_tables,
    format_concatenated,
    format_weighted_summary,
)
from llm_tracker.file_handlers import load_human_coding
from llm_tracker.utils import format_coding_table

/Users/freymon.perez/Projects/GitHub/llm_tracker/src/llm_tracker/models.py:38: UserWarning: Field name "construct" in "ConstructInstance" shadows an attribute in parent "BaseModel"
  class ConstructInstance(BaseModel):


## Paths and Configuration

Set these values before running the rest of the tutorial. The sample CSV and codebook are bundled with the repo.

- `api_key` - your OpenRouter API key, **or** the path to a `.env` file containing it. OpenRouter API keys are case-sensitive.
- `llm_model` - any model listed at https://openrouter.ai/models. The same model is used for LLM coding and comparison matching.
- `input_data` - the input CSV (one document per row).
- `codebook_path` - JSON file defining the constructs the LLM should look for.
- `fuzzy_quote_matching` - optional quote-index recovery. By default this is `False`, so quote indices are assigned only when the LLM quote appears exactly in the source text. Set it to `True` if you want slower fuzzy recovery for near-exact quotes with small punctuation or spacing changes.


In [2]:
api_key = "/Users/freymon.perez/Projects/GitHub/llm_tracker/openrouter_api.env"
llm_model = "google/gemini-3-flash-preview"
input_data = "sample_data/reddit_autism_anxiety_depression.csv"
codebook_path = "/Users/freymon.perez/Projects/GitHub/llm_tracker/codebooks/codebook_2_4_2025.json"
fuzzy_quote_matching = False  # set True for slower fuzzy quote-index recovery
temperature = 0  # 0 = deterministic/reproducible; raise toward 1.0 for more variety

# Shared configuration. Passing config= keeps the analyzer and comparer settings
# in one place. (When config= is given, individual kwargs are ignored.)
config = AnalyzerConfig(
    api_key=api_key,
    model_name=llm_model,
    fuzzy_quote_matching=fuzzy_quote_matching,
    temperature=temperature,
)


---
# Part 0 - Text Descriptives.(Optional)

Before we begin with the pipline we'll get some general information about the text we're working with. We'll use https://github.com/HLasse/TextDescriptives/tree/main?tab=readme-ov-file to accomplish this. 

Text Descriptives will give us a number of metrics to gain an understanding of the structure of our text data. Please note that Text descriptives works on a per-document basis, and does not accept directories of documents. As such if you have several documents you want metrics on you will need to run them individually or concatenate them prior to using the package.

For the purposes of this tutorial we'll use the reddit sample data here, but feel free to change the path to your own data.

In [ ]:
# Download the small English spaCy model (used by TextDescriptives for tokenization & parsing)
# If you'd like to use bigger models than en_core_web_sm (12 MB) optiosn include: 
# en_core_web_md (40 MB); en_core_web_lg (560 MB); en_core_web_trf (440 MB- Most accurate but slowest)
!python -m spacy download en_core_web_sm

In [ ]:
!pip install sentence-transformers

In [3]:
import pandas as pd
from llm_tracker.corpus_summary import print_summary, summarize_corpus

# Load the Reddit sample data
reddit_data = pd.read_csv("sample_data/reddit_autism_anxiety_depression.csv")

# Compute corpus-level descriptive metrics
summary, raw = summarize_corpus(
    reddit_data["post"].tolist(),
    spacy_model="en_core_web_sm",
    include_readability=True,
    include_coherence_sbert=True
)

print_summary(summary)

Extracting TextDescriptives metrics (descriptive_stats, pos_proportions, readability)…


/Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading SBERT model 'all-MiniLM-L6-v2'…


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10528.44it/s]
/Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Computing SBERT coherence per document…

Metric                                       Value
N (documents)                                60
Sentences per document                       mean 9.12 (SD 7.32) [min 1.00 – max 32.00]
Sentence length (words)                      mean 18.93 (SD 13.88) [min 6.44 – max 112.00]
Tokens per document                          mean 150.60 (SD 113.99) [min 16.00 – max 489.00]
Unique tokens per document                   mean 87.78 (SD 51.47) [min 13.00 – max 220.00]
Type-token ratio (TTR)                       mean 0.66 (SD 0.13) [min 0.44 – max 0.94]
Characters per document                      mean 619.32 (SD 470.07) [min 57.00 – max 2163.00]
1st-person pronoun proportion                mean 10.50% (SD 4.78%) [min 0.00% – max 21.62%]
Proportion NOUN                              mean 13.27% (SD 4.31%) [min 0.00% – max 22.58%]
Proportion VERB                              mean 14.55% (SD 3.07%) [min 8.70% – max 25.00%]
Proportion ADJ                    

### How to read the summary table

Every row summarizes one metric **across all documents in your corpus** 
**`mean (SD) [min–max]`**

- **mean** — the average value across documents
- **(SD)** — the standard deviation across documents, in parentheses
- **[min–max]** — the smallest and largest document values, in brackets

So `Sentences per document  9.12 (7.32) [1.00–32.00]` means: documents average 9.12 sentences (SD 7.32), ranging from 1 to 32.
documents the SD and range reflect individual documents rather than a stable distribution.

### What each metric measures

| Metric | What it measures |
|---|---|
| N (documents) | Number of documents summarized (count, not an average). |
| Sentences per document | How many sentences each document contains. |
| Sentence length (words) | Average words per sentence. |
| Tokens per document | Total tokens (words and punctuation) per document. |
| Unique tokens per document | Distinct tokens per document. |
| Type-token ratio (TTR) | Lexical diversity: unique tokens ÷ total tokens (0–1; higher = more varied vocabulary). |
| Characters per document | Total characters per document. |
| 1st-person pronoun proportion | Share of tokens that are first-person pronouns (I, me, we, …). |
| Proportion NOUN / VERB / ADJ / ADV / PRON | Share of tokens of each part of speech. |
| Flesch Reading Ease | Readability on a ~0–100 scale; **higher = easier** to read. |
| Flesch–Kincaid Grade | Readability as an approximate US school grade level; **higher = harder**. |
| Gunning Fog Index | Grade level needed to understand the text on first reading; higher = harder. |
| Coleman–Liau Index | Grade-level readability based on characters per word and sentences; higher = harder. |
| Automated Readability Index (ARI) | Grade-level readability based on characters and word/sentence counts; higher = harder. |
| SMOG | Grade level estimated from polysyllable counts; higher = harder. |
| Semantic coherence — SBERT | Mean cosine similarity between consecutive sentence embeddings (0–1); higher = sentences flow together more smoothly. |

> The readability indices each estimate difficulty differently, so their numbers will not agree exactly — they are meant to be read as a cluster, not compared one-to-one.


---

# Part 1 — Dummy Data Workflow

In this part we run the LLM twice over the same sample CSV and treat the second run as a stand-in for human coding. This lets you test the full pipeline end-to-end before you have real human-coded data to compare against.

Because the same model is run twice on the same documents, agreement will be high but not perfect — the LLM introduces some variability between runs, which makes this a reasonable functional test of the whole workflow.

## Step 1 (Dummy): LLM Coding

This step runs the LLM over every row of `csv_path` and extracts quotes that are instances of each construct in the codebook. Each document is processed independently, and results are saved to a timestamped output directory so runs are never overwritten.

The output directory has the following structure:
```
LLM_coding_YYYY-MM-DD_HHMMSS/
    encodings/     <- one JSON per document, containing extracted construct instances
    metadata/    <- token usage, latency, and model info per document
    errors/      <- any documents that failed, with error messages
    README.md
```

Each coding JSON contains a list of instances, where each instance includes the construct name, the extracted quote, character-level indices into the source text, and a confidence score from the LLM.

In [ ]:
analyzer = LLMTrackerAnalyzer(config=config)

results_llm, metadata_llm, errors_llm = analyzer.analyze_csv(
    csv_path=input_data,
    text_column="post",
    id_column="author",  # human-readable IDs; omit to use row index
    codebook_path=codebook_path,
    output_dir="LLM_coding",  # timestamp suffix added automatically
)

### Alternative: directory of `.txt` files

If your data is a folder of `.txt` files (one document per file) instead of a CSV, use `analyze_directory(...)` instead of `analyze_csv(...)`. Same return shape, same downstream pipeline. Each filename (without extension) becomes the `document_id`. The cell below is left commented out — uncomment and set `input_dir` if that's your input format.

In [4]:
# Alternative: analyze a directory of .txt files instead of a CSV.

# input_dir = ""  # path to your directory of .txt files

# results_llm, metadata_llm, errors_llm = analyzer.analyze_directory(
#     input_dir=input_dir,
#     codebook_path=codebook_path,
#     output_dir="LLM_coding",  # timestamp suffix added automatically
# )

The analyzer always returns and saves three dictionaries:
- `results` — what the LLM actually coded
- `metadata` — information about how the LLM was prompted (tokens, latency, model)
- `errors` — information on any calls that failed

These are also persisted as JSON files inside the timestamped output directory.

In [5]:
format_coding_table(results_llm)

Coding Table
------------
One row per construct instance across all documents. Each row represents
a single quote identified by the LLM as an instance of a construct.

Columns:
  doc_id      : document identifier (filename or subreddit_author)
  construct   : psychological construct the instance belongs to
  quote       : exact quote extracted from the source text
  speaker_id  : speaker identifier if available in the source text
  quote_index : character-level start:end indices of the quote
  confidence  : LLM confidence score (0=not mentioned/negated,
                1=indirect, 2=clear)



,doc_id,construct,quote,speaker_id,quote_index,confidence
0,anxiety_An0nus3r123,daily_functioning,I know my anxiety has made it hard for me to g...,None,71:138,2
1,anxiety_An0nus3r123,social_relationships_negative,had a hard time being around new people,None,30:69,2
2,anxiety_An0nus3r123,social_relationships_negative,When i talk to strangers it's one they I'll pr...,None,139:255,2
3,anxiety_An0nus3r123,social_relationships_negative,anxiety can make it hard for people to go out ...,None,453:513,2
4,anxiety_An0nus3r123,social_relationships_negative,"i feel like saying watching tv, reading, fishi...",None,None,1
...,...,...,...,...,...,...
518,depression_Domihoes,social_relationships_negative,been forever alone forever,None,337:363,2
519,depression_Domihoes,suicide_ideation,i dont know how i should go on in life anymore,None,430:476,1
520,depression_Domihoes,suicide_reason_dying,with no close family members this life seems p...,None,482:536,2
521,depression_Domihoes,understanding_mental_illness,i have deep depression,None,44:66,2


### Retry Failed Documents (optional)

If any documents failed — due to API errors, rate limits, or malformed responses — you can retry only the failed ones without re-processing documents that already succeeded. Uncomment the code below and update `output_dir` to point to the timestamped directory from your run above.

In [6]:
# recovered_results, recovered_meta, remaining_errors = analyzer.retry_errors(
#     output_dir="path/to/LLM_coding_YYYY-MM-DD_HHMMSS",
#     codebook_path=codebook_path,
# )

## Step 2 (Dummy): Stand-in Human Coding via a Second LLM Run

For the comparison step we need a second set of codings. In Part 1 we generate that by running the LLM a second time over the same documents — the variability between runs gives us something non-trivial to compare against.

Variables in this section are prefixed with `dummy_human_` to make it clear this is *not* real human data.

> If you have real human codings, **skip Part 1 entirely and jump to Part 2**, which shows the typical workflow.

In [ ]:
dummy_human_results, dummy_human_metadata, dummy_human_errors = analyzer.analyze_csv(
    csv_path=input_data,
    text_column="post",
    id_column="author",  # human-readable IDs; omit to use row index
    codebook_path=codebook_path,
    output_dir="dummy_human_coding",  # timestamp suffix added automatically
)

In [8]:
format_coding_table(dummy_human_results)

Coding Table
------------
One row per construct instance across all documents. Each row represents
a single quote identified by the LLM as an instance of a construct.

Columns:
  doc_id      : document identifier (filename or subreddit_author)
  construct   : psychological construct the instance belongs to
  quote       : exact quote extracted from the source text
  speaker_id  : speaker identifier if available in the source text
  quote_index : character-level start:end indices of the quote
  confidence  : LLM confidence score (0=not mentioned/negated,
                1=indirect, 2=clear)



,doc_id,construct,quote,speaker_id,quote_index,confidence
0,anxiety_An0nus3r123,problem_recognition,I know my anxiety has made it hard for me to g...,None,71:138,2
1,anxiety_An0nus3r123,social_relationships_negative,had a hard time being around new people,None,30:69,2
2,anxiety_An0nus3r123,social_relationships_negative,When i talk to strangers it's one they I'll pr...,None,139:255,2
3,anxiety_An0nus3r123,social_relationships_negative,anxiety can make it hard for people to go out ...,None,453:513,2
4,anxiety_An0nus3r123,social_relationships_negative,idk how to come off as a fun person and that i...,None,688:801,2
...,...,...,...,...,...,...
504,depression_Domihoes,social_relationships_negative,been forever alone forever,None,337:363,2
505,depression_Domihoes,suicide_ideation,i dont know how i should go on in life anymore,None,430:476,1
506,depression_Domihoes,suicide_reason_dying,with no close family members this life seems p...,None,482:536,2
507,depression_Domihoes,understanding_mental_illness,i have deep depression,None,44:66,2


## Step 3 (Dummy): Comparison

This step aligns the dummy-human and LLM codings for each document. For each construct, an LLM matcher reads all quotes from both coders and decides which dummy-human quotes and LLM quotes refer to the same passage or idea — even if the wording differs (paraphrase).

Each instance in the resulting DataFrame is classified as one of three statuses:
- **`matched`** — both coders identified this passage (True Positive, TP). Includes a `match_confidence` score from the matcher and a `span_overlap` score (Jaccard overlap of character indices).
- **`human_only`** — the (dummy) human identified this passage but the LLM missed it (False Negative, FN).
- **`llm_only`** — the LLM identified this passage but the (dummy) human missed it (False Positive, FP).

You can optionally use a different model for the matching step than for coding — for example, a more capable model for matching if accuracy is critical. Set `match_model` accordingly.

In [9]:
comparator = LLMTrackerComparer(config=config)

comparison_table = comparator.compare_results(
    human_results=dummy_human_results,
    llm_results=results_llm,
    output_dir="dummy_comparison_run",  # optional; saves comparison_rows.csv
)

Task exception was never retrieved
future: <Task finished name='Task-614' coro=<AsyncClient.aclose() done, defined at /Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/httpx/_client.py:2024> exception=RuntimeError('Event loop is closed')>
Traceback (most recent call last):
  File "/Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/httpx/_client.py", line 2031, in aclose
    await self._transport.aclose()
  File "/Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 389, in aclose
    await self._pool.aclose()
  File "/Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 353, in aclose
    await self._close_connections(closing_connections)
  File "/Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 345, in _close

## Step 4 (Dummy): Build Summary Tables

Now that we have compared the LLM coding to the dummy-human coding, we can produce summary tables that make the results easier to read.

These tables answer slightly different questions:

- **Concatenated summary:** if we pool all coded examples across all documents together, how well did the LLM perform for each construct?
- **Per-document summary:** how well did the LLM perform within each individual document?
- **Weighted summary:** what does performance look like across documents when documents with more coded material count more heavily?

### Important note about the metrics

This is an open-ended coding task, not a standard classification task with true negatives.

We observe:
- True positives (TP)
- False positives (FP)
- False negatives (FN)

But we do **NOT** have true negatives (TN), so:
- Specificity is not meaningful
- ROC AUC is not appropriate

**A note on Cohen's Kappa.** Cohen's Kappa adjusts agreement for chance, but it requires a true-negative count. Because open-ended span coding has no observable true negatives, kappa is not a good fit for this pipeline. The summary tables report **PABAK** (prevalence-adjusted bias-adjusted kappa) instead.

Reported metrics: precision, recall (sensitivity), F1, PABAK, and PR AUC.

### 4.1 Row-Level Comparison DataFrame

The raw DataFrame below has one row per coded instance across all documents. Each row represents a single quote that was identified by at least one coder, with the full quote text, character indices, and TP/FP/FN indicators. This is the most granular view and is useful for inspecting specific mismatches — for example, filtering to `status == 'llm_only'` to review what the LLM found that the dummy human missed.

In [10]:
comparison_table

,doc_id,construct,status,human_quote,llm_quote,human_indices,llm_indices,human_confidence,llm_confidence,paraphrase,span_overlap,match_confidence,tp,fp,fn
0,anxiety_4ssCr4ck,daily_functioning,matched,i have always felt like that and it has affect...,i have always felt like that and it has affect...,460:548,460:548,2.0,2.0,False,1.0,1.0,1,0,0
1,anxiety_4ssCr4ck,physical_health_symptoms,matched,"i also began to sweat, shake, i felt like thro...","i also began to sweat, shake, i felt like thro...",314:367,314:367,2.0,2.0,False,1.0,1.0,1,0,0
2,anxiety_4ssCr4ck,physical_health_symptoms,matched,i could feel something in my chest,i could feel something in my chest,387:421,387:421,2.0,2.0,False,1.0,1.0,1,0,0
3,anxiety_4ssCr4ck,problem_recognition,matched,Is this anxiety? I know this might be a dumb q...,Is this anxiety? I know this might be a dumb q...,0:171,0:171,2.0,2.0,False,1.0,1.0,1,0,0
4,anxiety_4ssCr4ck,problem_recognition,matched,i am asking this question because of something...,i am asking this question because of something...,173:277,173:277,2.0,2.0,False,1.0,1.0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,depression_scrappydoosghost,suicide_reason_dying,llm_only,None,So many of my friends have left,None,375:406,NaN,2.0,None,NaN,NaN,0,1,0
535,depression_scrappydoosghost,suicide_reason_not_dying,matched,"I wish I had the guts, the motivation, to kill...","I wish I had the guts, the motivation, to kill...",232:286,232:286,2.0,2.0,False,1.0,1.0,1,0,0
536,depression_thenerdycatmom,care_encounter_description,matched,My doctor decided to add a medication,My doctor decided to add a medication,43:80,43:80,2.0,2.0,False,1.0,1.0,1,0,0
537,depression_thenerdycatmom,care_encounter_unpleasant,matched,I've had this one before but the side effects ...,I've had this one before but the side effects ...,172:229,172:229,2.0,2.0,False,1.0,1.0,1,0,0


In [11]:
per_interview, concatenated, weighted_summary = compute_summary_tables(comparison_table)

### 4.2 Concatenated Metrics (primary view)

One row per construct, with counts pooled across **all documents** before metrics are computed. Think of this as treating the entire dataset as a single document. This table answers: *overall, across all documents, how did the LLM perform on each construct?*

The `interviews_with_construct [p5-p95]` column shows how many documents contained at least one instance of the construct, along with the 5th and 95th percentile of instance counts — giving a sense of how common and variable the construct is across your data.

In [12]:
format_concatenated(concatenated)

,construct,tp,fp,fn,sensitivity,precision,f1,pabak,pr_auc,interviews_with_construct [p5-p95]
0,care_encounter_description,8,1,0,1.00,0.89,0.94,0.78,0.89,9 [0.00-1.00]
1,care_encounter_ending,1,0,0,1.00,1.00,1.00,1.00,-,1 [0.00-0.00]
2,care_encounter_unpleasant,2,0,0,1.00,1.00,1.00,1.00,-,2 [0.00-0.00]
3,daily_functioning,23,1,0,1.00,0.96,0.98,0.92,0.96,15 [0.00-2.05]
4,diet_exercise,9,0,0,1.00,1.00,1.00,1.00,-,7 [0.00-1.00]
5,finances,7,0,0,1.00,1.00,1.00,1.00,-,6 [0.00-1.00]
6,guilt_shame,14,2,4,0.78,0.88,0.82,0.40,0.93,13 [0.00-2.05]
7,healthcare_system,1,0,0,1.00,1.00,1.00,1.00,-,1 [0.00-0.00]
8,media,9,0,0,1.00,1.00,1.00,1.00,-,5 [0.00-1.00]
9,mental_health_beliefs,1,0,0,1.00,1.00,1.00,1.00,-,1 [0.00-0.00]


### Alternate Summary Tables

The two tables below offer complementary views to the concatenated summary above. Use them when you need to dig into individual-document performance or want results that down-weight documents with very little coded material.

#### Per-Document Metrics

One row per (document, construct) combination. Shows raw TP/FP/FN counts and all metrics broken down by document. Use this to identify whether performance issues are consistent across documents or driven by specific ones. Constructs that do not appear in a given document are absent from this table, they do not appear as zero rows. 

In [13]:
per_interview

,doc_id,construct,tp,fp,fn,union,sensitivity,precision,f1,pabak,pr_auc
0,anxiety_4ssCr4ck,daily_functioning,1,0,0,1,1.0,1.0,1.0,1.0,NaN
1,anxiety_4ssCr4ck,physical_health_symptoms,2,0,0,2,1.0,1.0,1.0,1.0,NaN
2,anxiety_4ssCr4ck,problem_recognition,2,0,0,2,1.0,1.0,1.0,1.0,NaN
3,anxiety_4ssCr4ck,stress_response,1,0,0,1,1.0,1.0,1.0,1.0,NaN
4,anxiety_4ssCr4ck,understanding_mental_illness,1,0,0,1,1.0,1.0,1.0,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
384,depression_scrappydoosghost,suicide_reason_dying,0,4,0,4,NaN,0.0,0.0,-1.0,NaN
385,depression_scrappydoosghost,suicide_reason_not_dying,1,0,0,1,1.0,1.0,1.0,1.0,NaN
386,depression_thenerdycatmom,care_encounter_description,1,0,0,1,1.0,1.0,1.0,1.0,NaN
387,depression_thenerdycatmom,care_encounter_unpleasant,1,0,0,1,1.0,1.0,1.0,1.0,NaN


#### Weighted Summary

One row per construct. Metrics are first computed per document, then summarized as **weighted median [min–max]** across documents, where each document is weighted by its union size (TP + FP + FN) — so documents with more coded instances contribute more to the median. This table answers: *what does performance typically look like at the individual document level?* The `[min–max]` brackets show the range across documents, useful for spotting constructs with highly variable performance.



In [14]:
format_weighted_summary(weighted_summary)

,construct,tp,fp,fn,sensitivity,precision,f1,pabak,pr_auc,interviews_with_construct [p5-p95]
0,daily_functioning,23,1,0,1.00 [1.00-1.00],1.00 [0.67-1.00],1.00 [0.80-1.00],1.00 [0.33-1.00],0.67 [0.67-0.67],15 [0.00-2.05]
1,physical_health_symptoms,10,0,1,1.00 [0.00-1.00],1.00 [1.00-1.00],1.00 [0.00-1.00],1.00 [-1.00-1.00],-,10 [0.00-1.00]
2,problem_recognition,17,1,1,1.00 [0.00-1.00],1.00 [0.00-1.00],1.00 [0.00-1.00],1.00 [-1.00-1.00],-,16 [0.00-1.05]
3,stress_response,19,0,0,1.00 [1.00-1.00],1.00 [1.00-1.00],1.00 [1.00-1.00],1.00 [1.00-1.00],-,15 [0.00-1.05]
4,understanding_mental_illness,26,1,1,1.00 [0.00-1.00],1.00 [0.00-1.00],1.00 [0.00-1.00],1.00 [-1.00-1.00],-,24 [0.00-1.05]
5,self_identity,20,2,1,1.00 [0.50-1.00],1.00 [0.00-1.00],1.00 [0.00-1.00],1.00 [-1.00-1.00],0.50 [0.50-0.50],17 [0.00-2.00]
6,social_relationships_negative,35,3,0,1.00 [1.00-1.00],1.00 [0.00-1.00],1.00 [0.00-1.00],1.00 [-1.00-1.00],1.00 [0.50-1.00],24 [0.00-2.00]
7,social_relationships_other,3,1,0,1.00 [1.00-1.00],0.50 [0.50-1.00],0.67 [0.67-1.00],0.00 [0.00-1.00],0.50 [0.50-0.50],3 [0.00-0.05]
8,physical_environment,9,1,1,1.00 [0.50-1.00],1.00 [0.50-1.00],1.00 [0.50-1.00],1.00 [-0.33-1.00],0.50 [0.50-0.50],6 [0.00-1.05]
9,self_management_challenges,13,0,0,1.00 [1.00-1.00],1.00 [1.00-1.00],1.00 [1.00-1.00],1.00 [1.00-1.00],-,10 [0.00-1.05]


---

# Part 1.5 — Codebook Optimization (testing)

Take the constructs the LLM coded worst (PABAK below a threshold) and fold their
disagreements back into the codebook as new `examples` (false negatives) and
`counter_examples` (false positives). `optimize_codebook` outputs a **partial**
codebook of just the changed constructs; `merge_codebooks` applies a chosen
partial onto the full codebook to produce a new versioned codebook.

By default up to **50** new examples and **50** counter-examples are added per construct (a random, seeded sample when there are more). Pass an integer to change the cap, or `"all"` to add every false negative / false positive.

In [ ]:
from llm_tracker.comparison import merge_codebooks, optimize_codebook
from llm_tracker.file_handlers import load_codebook

current_codebook = load_codebook(codebook_path)

# One optimization pass (rerun_optimized_codebook=0 -> no re-coding, just v001).
partials = optimize_codebook(
    comparison_df=comparison_table,
    concatenated_summary=concatenated,
    codebook=current_codebook,
    human_results=dummy_human_results,
    analyzer=analyzer,
    csv_path=input_data,
    analyze_kwargs={"text_column": "post"},
    base_name="reddit_demo",
    output_dir="optimized_codebooks",
    n_examples=50,          # per construct: positive int, or "all" to add every FN
    n_counterexamples=50,   # per construct: positive int, or "all" to add every FP
    seed=0,                 # reproducible sampling
    rerun_optimized_codebook=0,
)

print(f"{len(partials)} partial(s) produced.")

In [ ]:
# Inspect what changed in the most recent partial.
if partials:
    latest = partials[-1]["codebook"]
    for construct, entry in latest.items():
        n_ex = len(entry.get("examples", []))
        n_ct = len(entry.get("counter_examples", []))
        print(f"{construct}: {n_ex} examples, {n_ct} counter-examples")
else:
    print("No constructs were below the PABAK threshold; nothing to merge.")

In [ ]:
# Merge a chosen partial back onto the full codebook (replaces those constructs,
# mints a new version). Pick whichever version you want, e.g. partials[0] = v001.
if partials:
    merged_codebook = merge_codebooks(
        base=current_codebook,
        partial=partials[0],
        output_path="reddit_demo_merged_codebook.json",
    )
    print("Merged codebook version:", merged_codebook["metadata"]["version"])

---

# Part 2 — Real Human Data Workflow

In a typical research workflow you will compare the LLM's coding against **real human-coded data**, not against a second LLM run. This part walks through that workflow end-to-end.

The steps are the same as Part 1, with one key difference: instead of generating a stand-in human coding via a second LLM run, you load human-coded data from a CSV or xlsx file (e.g., a Dedoose excerpt export, or any other tabular human-coding format) using `load_human_coding(...)`.

## Step 1: LLM Coding

This is the same LLM-coding step as in Part 1.

> **If you already ran the LLM in Part 1 and want to reuse those results**, skip this cell and continue to Step 2 — Part 2 below uses `results_llm` from above.
>
> **Run this cell only if you want a fresh, independent LLM run for Part 2** (e.g., a different model, a different output directory, or you skipped Part 1 entirely).

In [ ]:
input_data = "" # path to the csv or directory the llm will be coding.

In [ ]:
# Run this only if you want a fresh LLM run for Part 2; otherwise skip and reuse `results_llm` from Part 1.

real_analyzer = LLMTrackerAnalyzer(
    api_key=api_key,
    model_name=llm_model,
    fuzzy_quote_matching=fuzzy_quote_matching,
)

results_llm, metadata_llm, errors_llm = real_analyzer.analyze_csv(
    csv_path=input_data,
    text_column="post",
    id_column="author",  # human-readable IDs; omit to use row index
    codebook_path=codebook_path,
    output_dir="LLM_coding_real",  # timestamp suffix added automatically
)
format_coding_table(results_llm)

### Alternative: directory of `.txt` files

Same as in Part 1 — if your input is a folder of `.txt` files, swap `analyze_csv` for `analyze_directory`. Cell below left commented as a reference.

In [ ]:
# Alternative: analyze a directory of .txt files instead of a CSV.

# input_dir = ""  # path to your directory of .txt files

# results_llm, metadata_llm, errors_llm = real_analyzer.analyze_directory(
#     input_dir=input_dir,
#     codebook_path=codebook_path,
#     output_dir="LLM_coding_real",  # timestamp suffix added automatically
# )

## Step 2 : Loading Human-Coded Data

Real human coding typically lives in a tabular file (CSV, TSV, or xlsx) where each row is a single coded excerpt. `load_human_coding(...)` reads that file and returns the same shape as `analyze_csv` / `analyze_directory`: a dictionary mapping document IDs to `AnalysisResult` objects. That means it can be passed straight to the comparator.

The loader is **generic** — it works with any tabular human-coding source as long as you tell it which columns to use:

| Argument            | Default                       | What it identifies                                                    |
|---------------------|-------------------------------|------------------------------------------------------------------------|
| `doc_id_col`        | `"Media Title"`               | Column whose value matches the LLM `document_id` (filename / CSV row) |
| `quote_col`         | `"Excerpt Copy"`              | The verbatim text excerpt the human selected                           |
| `range_col`         | `"Excerpt Range"`             | Character-level start–end indices into the source document             |
| `construct_col`     | `"Codes Applied Combined"`    | Code(s) applied to the excerpt; multiple codes are split on the separator |
| `range_format`      | `"dash"` (e.g. `"858-1159"`)  | `"dash"` for Dedoose-style; `"colon"` for `start:end` format           |
| `construct_separator` | `","`                       | Used to split multi-code cells into individual constructs              |

The defaults match a Dedoose **Excerpt Export** out of the box, so a Dedoose user only needs `load_human_coding(path)`. For other tabular sources, override the column-name kwargs accordingly.

> **Important — document-id alignment.** The values in `doc_id_col` must match the document IDs used in your LLM run (typically the source filename without extension, or the CSV `id_column` value — or the row index if no `id_column` was given). If they don't align, the comparator won't pair instances correctly.

In [ ]:
human_csv_path = ""  # path to your human-coded CSV/xlsx export

# Generic example — override only the kwargs that differ from the Dedoose defaults.
# For a non-Dedoose source, pass the column names explicitly, e.g.:
#
# human_results = load_human_coding(
#     human_csv_path,
#     doc_id_col="participant_id",
#     quote_col="excerpt_text",
#     range_col="char_range",
#     construct_col="code",
#     range_format="colon",          # if your ranges look like "858:1159"
#     construct_separator=";",        # if codes are separated by something else
# )

human_results = load_human_coding(human_csv_path)

## Step 3: Comparison

Same comparison step as Part 1, but now we pass the real human results in place of the dummy ones.

In [ ]:
real_comparator = LLMTrackerComparer(
    api_key=api_key,
    match_model=llm_model,
)

comparison_table_real = real_comparator.compare_results(
    human_results=human_results,
    llm_results=results_llm,
    output_dir="real_comparison_run",  # optional; saves comparison_rows.csv
)

## Step 4: Summary Tables

Same four summary tables as Part 1, in the same order: row-level table → concatenated → alternate (per-document + weighted). See the metrics note in Part 1 Step 4 for details on PABAK vs Cohen's Kappa.

### 4.1 Row-Level Comparison DataFrame

One row per coded instance across all documents — useful for inspecting specific mismatches.

In [ ]:
comparison_table_real

In [ ]:
per_interview_real, concatenated_real, weighted_summary_real = compute_summary_tables(comparison_table_real)

### 4.2 Concatenated Metrics (primary view)

Counts pooled across all documents — overall LLM performance per construct against your real human coders.

In [ ]:
format_concatenated(concatenated_real)

### Alternate Summary Tables

#### Per-Document Metrics

Performance broken down by document — useful for spotting documents that drive overall numbers.

In [ ]:
per_interview_real

#### Weighted Summary

Per-document metrics summarized as weighted median [min–max] across documents.

In [ ]:
format_weighted_summary(weighted_summary_real)